In [ ]:
import sys
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight


sys.path.append("../..")
from utils.seed import set_seed
from utils.config import save_run_manifest
from utils.split import build_or_load_manifest, print_split_summary
from utils.dataset import ManifestImageDataset, get_manifest_dataloaders
from utils.engine import evaluate, run_phase
from utils.features import build_feature_extractor, extract_and_cache_embeddings
from utils.classifier import tune_svm_hyperparams
from utils.submission import (
    build_class_index_mapping,
    load_test_images,
    print_label_distribution,
)

warnings.filterwarnings("ignore")

# **EXPERIMENT CONFIG**

In [ ]:
EXPERIMENT_CONFIG = {
    "seed": 42,
    "image_size": (224, 224),
    "batch_size": 32,
    "augmentation": {
        "rotation_range": 20,
        "shift_range": 0.2,
        "shear_range": 0.2,
        "zoom_range": 0.2,
        "brightness_range": [0.8, 1.2],
    },
    "model": {
        "backbone": "convnext_tiny",
        "fine_tune_at": 147,
    },
    "split": {
        "holdout_ratio": 0.20,
        "inner_val_ratio": 0.15,
    },
    "training": {
        "phase1_epochs": 10,
        "phase2_epochs": 10,
        "phase1_lr": 1e-3,
        "phase2_lr": 1e-5,
        "early_stop_patience_phase1": 2,
        "early_stop_patience_phase2": 5,
        "lr_reduce_factor": 0.5,
        "lr_reduce_patience": 3,
        "min_lr": 1e-7,
    },
    "svm": {
        "kernel": "rbf",
        "C_grid": [0.1, 1.0, 10.0, 100.0],
        "gamma_grid": ["scale", 0.01, 0.001],
        "n_folds": 5,
    },
}

In [ ]:
RANDOM_SEED = EXPERIMENT_CONFIG["seed"]
set_seed(RANDOM_SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")

# **DATA LOADER**

In [ ]:
TEST_DATA_DIR = Path("../../data/test")

RAW_FOLDER_TO_CLASS_NAME = {
    "0_Recyclable": "Recyclable",
    "1_Electronic": "Electronic",
    "2_Organic": "Organic",
}

CLASS_NAMES = sorted(RAW_FOLDER_TO_CLASS_NAME.values())
NUM_CLASSES = len(CLASS_NAMES)

test_image_count = len(list(TEST_DATA_DIR.glob("*.jpg")))
print(f"Kelas ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Total test gambar: {test_image_count}")

# **SPLIT (train / inner_val / holdout, anti-leakage)**

Beda dari notebook 03/04 (yang cuma 2 split train/val dari `data/processed`): di sini dipecah 3 -- `train` (fine-tune), `inner_val` (early-stopping), `holdout` (dites terakhir, tak pernah disentuh fine-tuning maupun fit SVM/scaler). Dibuat SEKALI dari `data/train` mentah, stratified, seed tetap, dan disimpan sebagai manifest CSV (index-only, tidak menduplikasi file) supaya konsisten tiap kali notebook dijalankan ulang.

In [ ]:
RAW_DATA_DIR = Path("../../data/train")
MANIFEST_PATH = Path("../../data/splits/manifest.csv")
SPLIT_CFG = EXPERIMENT_CONFIG["split"]

manifest = build_or_load_manifest(
    raw_dir=RAW_DATA_DIR,
    class_names=RAW_FOLDER_TO_CLASS_NAME,
    manifest_path=MANIFEST_PATH,
    holdout_ratio=SPLIT_CFG["holdout_ratio"],
    inner_val_ratio=SPLIT_CFG["inner_val_ratio"],
    seed=RANDOM_SEED,
)
print_split_summary(manifest)

In [ ]:
class_indices = {name: idx for idx, name in enumerate(CLASS_NAMES)}
idx_to_class = {v: k for k, v in class_indices.items()}

train_labels = (
    manifest.loc[manifest["split"] == "train", "class_name"]
    .map(class_indices)
    .to_numpy()
)
unique_classes = np.unique(train_labels)

weights = compute_class_weight(
    class_weight="balanced", classes=unique_classes, y=train_labels
)
CLASS_WEIGHTS = dict(zip(unique_classes.tolist(), weights.tolist()))

class_weights_tensor = torch.tensor(
    [CLASS_WEIGHTS[i] for i in range(NUM_CLASSES)],
    dtype=torch.float32,
).to(device)

print("Class weights (untuk handle imbalance):")
for idx, w in CLASS_WEIGHTS.items():
    print(f"  {idx_to_class[idx]:12} (idx={idx}): weight={w:.4f}")

# **PREPROCESSING & AUGMENTATION**

In [ ]:
AUGMENTATION_CFG = EXPERIMENT_CONFIG["augmentation"]
BATCH_SIZE = EXPERIMENT_CONFIG["batch_size"]
IMAGE_SIZE = EXPERIMENT_CONFIG["image_size"]

# convnext_tiny pretrained (in1k) pakai statistik normalisasi ImageNet standar
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

train_transform = A.Compose(
    [
        A.Resize(*IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent={
                "x": (
                    -AUGMENTATION_CFG["shift_range"],
                    AUGMENTATION_CFG["shift_range"],
                ),
                "y": (
                    -AUGMENTATION_CFG["shift_range"],
                    AUGMENTATION_CFG["shift_range"],
                ),
            },
            shear=(
                -AUGMENTATION_CFG["shear_range"] * 90,
                AUGMENTATION_CFG["shear_range"] * 90,
            ),
            scale=(
                1 - AUGMENTATION_CFG["zoom_range"],
                1 + AUGMENTATION_CFG["zoom_range"],
            ),
            rotate=(
                -AUGMENTATION_CFG["rotation_range"],
                AUGMENTATION_CFG["rotation_range"],
            ),
            mode=cv2.BORDER_REPLICATE,
            p=1.0,
        ),
        A.RandomBrightnessContrast(
            brightness_limit=AUGMENTATION_CFG["brightness_range"][1] - 1.0,
            contrast_limit=0.0,
            p=1.0,
        ),
        A.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
        ToTensorV2(),
    ]
)

eval_transform = A.Compose(
    [
        A.Resize(*IMAGE_SIZE),
        A.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
        ToTensorV2(),
    ]
)

In [ ]:
train_loader, inner_val_loader, holdout_loader = get_manifest_dataloaders(
    manifest,
    CLASS_NAMES,
    train_transform,
    eval_transform,
    BATCH_SIZE,
)
print(
    f"train={len(train_loader.dataset)}  inner_val={len(inner_val_loader.dataset)}  holdout={len(holdout_loader.dataset)}"
)

# **BUILD MODEL**

In [ ]:
class ConvNeXtWasteClassifier(nn.Module):
    def __init__(self, num_classes: int, pretrained: bool = True) -> None:
        super().__init__()
        self.backbone = timm.create_model(
            EXPERIMENT_CONFIG["model"]["backbone"],
            pretrained=pretrained,
            num_classes=0,
            global_pool="",
        )
        feat_channels = self.backbone.num_features

        self.custom_conv2d = nn.Conv2d(feat_channels, 256, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=False)
        self.custom_maxpool = nn.MaxPool2d(kernel_size=2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=False),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(inplace=False),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

        self._init_head_weights()

    def _init_head_weights(self) -> None:
        for module in [self.custom_conv2d, self.classifier]:
            for m in module.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(
                        m.weight, mode="fan_out", nonlinearity="relu"
                    )
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
                elif isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(
                        m.weight, mode="fan_out", nonlinearity="relu"
                    )
                    nn.init.zeros_(m.bias)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        """Everything up to the pooled 256-d feature vector -- used both by
        forward() (training) and by the embedding extractor (SVM phase),
        which calls this directly to skip self.classifier entirely."""
        x = self.backbone(x)  # (B, C, H, W) -- convnext output is channel-first
        x = self.relu(self.custom_conv2d(x))
        x = self.custom_maxpool(x)
        x = self.global_pool(x)
        return torch.flatten(x, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.forward_features(x))


model = ConvNeXtWasteClassifier(NUM_CLASSES).to(device)

for param in model.backbone.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,} / {total:,}")

# **TRAINING LOOP**

In [ ]:
MODEL_NAME = "05_convnext_embedding_svm"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M")
RUN_NAME = f"{MODEL_NAME}_{RUN_ID}"
RUN_DIR = Path("../../models") / MODEL_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = RUN_DIR / "model.pt"
HISTORY_PATH = RUN_DIR / "training_history.json"
CONFIG_PATH = RUN_DIR / "config.json"
EMBEDDINGS_DIR = RUN_DIR / "embeddings"

print(f"RUN_ID  : {RUN_ID}")
print(f"RUN_DIR : {RUN_DIR}")

In [ ]:
TRAINING_CONFIG = EXPERIMENT_CONFIG["training"]

history_combined = {"accuracy": [], "val_accuracy": [], "loss": [], "val_loss": []}

print("=" * 50)
print("PHASE 1: Feature Extraction (backbone frozen)")
print("=" * 50)

optimizer_phase1 = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=TRAINING_CONFIG["phase1_lr"],
)
best_val_acc_phase1 = run_phase(
    model,
    optimizer_phase1,
    TRAINING_CONFIG["phase1_epochs"],
    TRAINING_CONFIG["early_stop_patience_phase1"],
    history_combined,
    "Phase1",
    train_loader=train_loader,
    val_loader=inner_val_loader,
    class_weights_tensor=class_weights_tensor,
    device=device,
    lr_reduce_factor=TRAINING_CONFIG["lr_reduce_factor"],
    lr_reduce_patience=TRAINING_CONFIG["lr_reduce_patience"],
    min_lr=TRAINING_CONFIG["min_lr"],
)
phase1_epoch_count = len(history_combined["accuracy"])

In [ ]:
print("\n" + "=" * 50)
print("PHASE 2: Fine-tuning (unfreeze top backbone layers)")
print("=" * 50)

fine_tune_at = EXPERIMENT_CONFIG["model"]["fine_tune_at"]

backbone_params = list(model.backbone.named_parameters())
for _, param in backbone_params:
    param.requires_grad = True
for name, param in backbone_params[:fine_tune_at]:
    param.requires_grad = False

trainable_in_backbone = sum(1 for _, p in backbone_params if p.requires_grad)
print(
    f"Trainable param tensors di backbone: {trainable_in_backbone} / {len(backbone_params)}"
)

optimizer_phase2 = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=TRAINING_CONFIG["phase2_lr"],
)
best_val_acc_phase2 = run_phase(
    model,
    optimizer_phase2,
    TRAINING_CONFIG["phase2_epochs"],
    TRAINING_CONFIG["early_stop_patience_phase2"],
    history_combined,
    "Phase2",
    train_loader=train_loader,
    val_loader=inner_val_loader,
    class_weights_tensor=class_weights_tensor,
    device=device,
    lr_reduce_factor=TRAINING_CONFIG["lr_reduce_factor"],
    lr_reduce_patience=TRAINING_CONFIG["lr_reduce_patience"],
    min_lr=TRAINING_CONFIG["min_lr"],
)

torch.save(model.state_dict(), CHECKPOINT_PATH)
print(f"Model saved to {CHECKPOINT_PATH}")

with open(HISTORY_PATH, "w") as f:
    json.dump(history_combined, f)
print(f"History saved to {HISTORY_PATH}")

num_epochs_trained = len(history_combined["accuracy"])
best_val_accuracy = max(history_combined["val_accuracy"])
print(f"\nTotal epoch dilatih : {num_epochs_trained}")
print(f"Best val_accuracy   : {best_val_accuracy:.4f}")

# **EVALUATION**

In [ ]:
train_acc_hist = history_combined["accuracy"]
val_acc_hist = history_combined["val_accuracy"]
train_loss_hist = history_combined["loss"]
val_loss_hist = history_combined["val_loss"]

fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(16, 6))

ax_acc.plot(train_acc_hist, label="Train Accuracy", color="darkblue", linewidth=2)
ax_acc.plot(val_acc_hist, label="Inner-Val Accuracy", color="crimson", linewidth=2)
ax_acc.axvline(
    x=phase1_epoch_count - 1,
    color="gray",
    linestyle="-",
    linewidth=1,
    label="Fine-tune start",
)
ax_acc.set_title("Model Accuracy", fontsize=14, fontweight="bold")
ax_acc.set_xlabel("Epoch")
ax_acc.set_ylabel("Accuracy")
ax_acc.legend()
ax_acc.grid(True, alpha=0.3)

ax_loss.plot(train_loss_hist, label="Train Loss", color="darkblue", linewidth=2)
ax_loss.plot(
    val_loss_hist, label="Inner-Val Loss", color="crimson", linewidth=2, linestyle="--"
)
ax_loss.axvline(
    x=phase1_epoch_count - 1,
    color="gray",
    linestyle="-",
    linewidth=1,
    label="Fine-tune start",
)
ax_loss.set_title("Model Loss", fontsize=14, fontweight="bold")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Loss")
ax_loss.legend()
ax_loss.grid(True, alpha=0.3)

plt.suptitle(
    "Learning Curve (ConvNeXt Waste Classifier - PyTorch)",
    fontsize=16,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
final_criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
train_loss, train_accuracy = evaluate(model, train_loader, final_criterion, device)
val_loss, val_accuracy = evaluate(model, inner_val_loader, final_criterion, device)

print("=" * 50)
print("EVALUASI FINAL (train / inner_val)")
print("=" * 50)
print(f"Train      Accuracy : {train_accuracy * 100:.2f}%")
print(f"Inner-Val  Accuracy : {val_accuracy   * 100:.2f}%")
print(f"Train      Loss     : {train_loss:.4f}")
print(f"Inner-Val  Loss     : {val_loss:.4f}")

# **HOLDOUT EVALUATION (fine-tuned classifier, sanity check)**

Bukan skor final kompetisi -- skor final tetap dari SVM di atas embedding (bagian di bawah). Ini cuma sanity check: backbone yang baru di-fine-tune beneran belajar sebelum lanjut ke ekstraksi embedding.

In [ ]:
@torch.no_grad()
def predict_all(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in dataloader:
        images = images.to(device)
        outputs = model(images)
        all_preds.append(outputs.argmax(1).cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_preds), np.concatenate(all_labels)


holdout_ft_preds, holdout_ft_labels = predict_all(model, holdout_loader, device)

print("Classification Report (fine-tuned classifier, Holdout Set):")
print(
    classification_report(holdout_ft_labels, holdout_ft_preds, target_names=CLASS_NAMES)
)
print(f"Macro-F1: {f1_score(holdout_ft_labels, holdout_ft_preds, average='macro'):.4f}")

# **EMBEDDING EXTRACTION (freeze backbone+head, strip classifier)**

Hybrid dimulai di sini: `forward_features()` (backbone + custom conv head, TANPA `self.classifier`) dipanggil frozen (`eval()`+`no_grad()`) buat keluarin embedding 256-d. Semua split (termasuk `train`) diekstrak pakai `eval_transform` (tanpa augmentasi) -- augmentasi cuma buat fine-tuning, bukan buat representasi embedding. Di-cache ke `.npz` biar SVM experiment berikutnya gak perlu re-run backbone.

In [ ]:
feature_extractor = build_feature_extractor(model, CHECKPOINT_PATH, device)

train_eval_ds = ManifestImageDataset(manifest, "train", CLASS_NAMES, eval_transform)
train_eval_loader = torch.utils.data.DataLoader(
    train_eval_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4
)

EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

train_embeddings, train_labels_emb = extract_and_cache_embeddings(
    feature_extractor, train_eval_loader, device, EMBEDDINGS_DIR / "train.npz"
)
inner_val_embeddings, inner_val_labels_emb = extract_and_cache_embeddings(
    feature_extractor, inner_val_loader, device, EMBEDDINGS_DIR / "inner_val.npz"
)
holdout_embeddings, holdout_labels_emb = extract_and_cache_embeddings(
    feature_extractor, holdout_loader, device, EMBEDDINGS_DIR / "holdout.npz"
)

# train_dev = train + inner_val (semua yang boleh dilihat backbone selama fine-tuning)
train_dev_embeddings = np.concatenate([train_embeddings, inner_val_embeddings])
train_dev_labels = np.concatenate([train_labels_emb, inner_val_labels_emb])

print(f"train_dev embeddings: {train_dev_embeddings.shape}")
print(f"holdout   embeddings: {holdout_embeddings.shape}")

# **SVM HYBRID: StandardScaler + SVC (fit HANYA di train_dev)**

`C`/`gamma` dipilih via `StratifiedKFold` GridSearch **di dalam train_dev saja**. `holdout` baru disentuh setelah pipeline final selesai di-fit -- gak pernah dipakai buat `fit`/`fit_transform` apapun.

In [ ]:
SVM_CFG = EXPERIMENT_CONFIG["svm"]

svm_pipeline = tune_svm_hyperparams(
    train_dev_embeddings,
    train_dev_labels,
    C_grid=SVM_CFG["C_grid"],
    gamma_grid=SVM_CFG["gamma_grid"],
    kernel=SVM_CFG["kernel"],
    n_folds=SVM_CFG["n_folds"],
    seed=RANDOM_SEED,
)

# **EVALUASI FINAL: holdout_test (skor kompetisi jujur)**

In [ ]:
holdout_preds = svm_pipeline.predict(holdout_embeddings)

holdout_accuracy = (holdout_preds == holdout_labels_emb).mean()
holdout_macro_f1 = f1_score(holdout_labels_emb, holdout_preds, average="macro")

print("=" * 50)
print("EVALUASI FINAL (SVM di atas embedding, Holdout Set)")
print("=" * 50)
print(f"Holdout Accuracy : {holdout_accuracy * 100:.2f}%")
print(f"Holdout Macro-F1 : {holdout_macro_f1:.4f}\n")
print(
    classification_report(holdout_labels_emb, holdout_preds, target_names=CLASS_NAMES)
)

In [ ]:
conf_matrix = confusion_matrix(holdout_labels_emb, holdout_preds)
conf_matrix_normalized = (
    conf_matrix.astype("float") / conf_matrix.sum(axis=1)[:, np.newaxis]
)

fig, (ax_count, ax_norm) = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(
    conf_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ax=ax_count,
)
ax_count.set_title("Confusion Matrix (Count) - Holdout", fontweight="bold")
ax_count.set_xlabel("Predicted")
ax_count.set_ylabel("Actual")
ax_count.tick_params(axis="x", rotation=45)

sns.heatmap(
    conf_matrix_normalized,
    annot=True,
    fmt=".2f",
    cmap="Reds",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ax=ax_norm,
)
ax_norm.set_title("Confusion Matrix (Normalized) - Holdout", fontweight="bold")
ax_norm.set_xlabel("Predicted")
ax_norm.set_ylabel("Actual")
ax_norm.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# **INFERENCE**

In [ ]:
SAMPLES_PER_CLASS = 5  # 3 kelas x 5 = 15 total sample yang divisualisasi

holdout_rows = manifest[manifest["split"] == "holdout"]
holdout_samples = []
for cls_name in CLASS_NAMES:
    cls_rows = holdout_rows[holdout_rows["class_name"] == cls_name]
    chosen = cls_rows.sample(
        min(SAMPLES_PER_CLASS, len(cls_rows)), random_state=RANDOM_SEED
    )
    for _, row in chosen.iterrows():
        holdout_samples.append((Path(row["filepath"]), cls_name))

random.shuffle(holdout_samples)

sample_batch = torch.stack(
    [
        eval_transform(image=np.array(Image.open(p).convert("RGB")))["image"]
        for p, _ in holdout_samples
    ]
).to(device)

with torch.no_grad():
    sample_embeddings = feature_extractor(sample_batch).cpu().numpy()
sample_preds = svm_pipeline.predict(sample_embeddings)

fig, axes = plt.subplots(3, 5, figsize=(18, 11))
axes = axes.flatten()

for i, (img_path, true_label) in enumerate(holdout_samples):
    pred_label = CLASS_NAMES[sample_preds[i]]
    is_correct = true_label == pred_label

    axes[i].imshow(Image.open(img_path).convert("RGB"))
    axes[i].set_title(
        f"True: {true_label}\nPred: {pred_label}",
        color="green" if is_correct else "red",
        fontsize=9,
    )
    axes[i].axis("off")

plt.suptitle(
    "Sample Predictions - SVM on Embeddings (Hijau=Benar, Merah=Salah)",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

# **OPTIONAL HOOKS (belum dikerjakan, cuma placeholder)**

- Nested CV penuh (re-fine-tune backbone per outer fold): mahal, butuh loop split di luar `train_dev` untuk tiap outer fold, lalu ulangi seluruh Phase 1+2+embedding.
- Grid search `LinearSVC` vs RBF `SVC`: tinggal tambah cabang di `utils/classifier.py` (`build_svm_pipeline(kernel="linear")`) dan bandingkan CV score-nya dengan yang RBF.
- Ensembling classifier lain (mis. `LogisticRegression`, `RandomForest`) di atas embedding yang sama: fit terpisah di `train_dev_embeddings`, gabung prediksi (voting/averaging) sebelum evaluasi holdout.

# **SAVE CONFIG**

In [ ]:
EXPERIMENT_CONFIG["run_id"] = RUN_ID
EXPERIMENT_CONFIG["run_name"] = RUN_NAME
EXPERIMENT_CONFIG["model_name"] = "convnext_embedding_svm_hybrid"
EXPERIMENT_CONFIG["base_architecture"] = (
    "timm convnext_tiny (num_classes=0 feature extractor) + custom conv head, "
    "embedding = forward_features() (classifier Linear layers stripped for SVM)"
)
EXPERIMENT_CONFIG["input_shape"] = [3, IMAGE_SIZE[0], IMAGE_SIZE[1]]
EXPERIMENT_CONFIG["num_classes"] = NUM_CLASSES
EXPERIMENT_CONFIG["class_names"] = CLASS_NAMES
EXPERIMENT_CONFIG["class_indices"] = class_indices
EXPERIMENT_CONFIG["class_weights"] = {
    idx_to_class[int(k)]: v for k, v in CLASS_WEIGHTS.items()
}
EXPERIMENT_CONFIG["pytorch_version"] = torch.__version__
EXPERIMENT_CONFIG["device"] = str(device)
EXPERIMENT_CONFIG["dataset"] = {
    "manifest_path": str(MANIFEST_PATH),
    "train_samples": len(train_loader.dataset),
    "inner_val_samples": len(inner_val_loader.dataset),
    "holdout_samples": len(holdout_loader.dataset),
}
EXPERIMENT_CONFIG["training"]["epochs_trained_total"] = num_epochs_trained
EXPERIMENT_CONFIG["svm"]["best_params"] = {
    k: v
    for k, v in svm_pipeline.named_steps["svc"].get_params().items()
    if k in ("C", "gamma", "kernel")
}
EXPERIMENT_CONFIG["callbacks"] = {
    "early_stopping": {
        "monitor": "val_accuracy",
        "patience_phase1": TRAINING_CONFIG["early_stop_patience_phase1"],
        "patience_phase2": TRAINING_CONFIG["early_stop_patience_phase2"],
        "restore_best_weights": True,
    },
    "reduce_lr_on_plateau": {
        "monitor": "val_loss",
        "factor": TRAINING_CONFIG["lr_reduce_factor"],
        "patience": TRAINING_CONFIG["lr_reduce_patience"],
        "min_lr": TRAINING_CONFIG["min_lr"],
    },
}

results = {
    "best_val_accuracy": best_val_accuracy,
    "final_train_accuracy": train_accuracy,
    "final_val_accuracy": val_accuracy,
    "final_train_loss": train_loss,
    "final_val_loss": val_loss,
    "holdout_finetuned_macro_f1": float(
        f1_score(holdout_ft_labels, holdout_ft_preds, average="macro")
    ),
    "holdout_svm_accuracy": float(holdout_accuracy),
    "holdout_svm_macro_f1": float(holdout_macro_f1),
}

artifacts = {
    "model_path": str(CHECKPOINT_PATH),
    "training_history_path": str(HISTORY_PATH),
    "embeddings_dir": str(EMBEDDINGS_DIR),
}

manifest_json = save_run_manifest(EXPERIMENT_CONFIG, results, artifacts, CONFIG_PATH)
print(f"Config saved to: {CONFIG_PATH}")
print(json.dumps(manifest_json, indent=2))

# **SUBMISSION**

In [ ]:
COMPETITION_LABEL_TO_NAME = {
    0: "Recyclable",
    1: "Electronic",
    2: "Organic",
}
ENGLISH_TO_COMPETITION_LABEL = {
    name: label for label, name in COMPETITION_LABEL_TO_NAME.items()
}

CLASS_INDEX_TO_COMPETITION_LABEL = build_class_index_mapping(
    class_indices,
    ENGLISH_TO_COMPETITION_LABEL,
)

print("Mapping model index -> competition label:")
for model_idx, comp_label in sorted(CLASS_INDEX_TO_COMPETITION_LABEL.items()):
    print(
        f"  model idx {model_idx} ({idx_to_class[model_idx]:12}) -> competition label {comp_label} ({COMPETITION_LABEL_TO_NAME[comp_label]})"
    )

In [ ]:
submission_template = pd.read_csv("../../submission/template_submission.csv")
test_ids = submission_template["id"].tolist()
print(f"Total test IDs: {len(test_ids)}")


def eval_preprocess(image: np.ndarray) -> np.ndarray:
    return eval_transform(image=image)["image"].numpy()


test_array, valid_ids = load_test_images(
    TEST_DATA_DIR,
    test_ids,
    eval_preprocess,
    IMAGE_SIZE,
)
print(f"Test array shape: {test_array.shape}")

test_batches = torch.from_numpy(test_array).float()
test_embeddings = []
with torch.no_grad():
    for i in range(0, len(test_batches), BATCH_SIZE):
        batch = test_batches[i : i + BATCH_SIZE].to(device)
        emb = feature_extractor(batch)
        test_embeddings.append(emb.float().cpu().numpy())
test_embeddings = np.concatenate(test_embeddings)

test_preds = svm_pipeline.predict(test_embeddings)
competition_preds = [CLASS_INDEX_TO_COMPETITION_LABEL[p] for p in test_preds]
competition_pred_names = [COMPETITION_LABEL_TO_NAME[p] for p in competition_preds]
print_label_distribution(
    competition_pred_names, list(COMPETITION_LABEL_TO_NAME.values())
)

In [ ]:
submission_df = pd.DataFrame(
    {
        "id": valid_ids,
        "predicted": competition_preds,
    }
)

submission_path = f"../../submission/submission_{MODEL_NAME}_{RUN_ID}.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(f"Total rows: {len(submission_df)}")
print("\nSample submission:")
print(submission_df.head(10).to_string(index=False))